# Your First PydanticAI Agent

**What you will build:** a solid grasp of the `Agent` object — the one thing you'll reuse in every remaining notebook of Block 1. System prompts (static and dynamic), running, and reading the result.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/11_first_pydanticai_agent.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — PydanticAI on OpenRouter. Run once per notebook.
!pip install -q "pydantic-ai-slim[openai]>=2.0,<3.0"   # verified against pydantic-ai 2.2 (2026)

import os, getpass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"   # any id from openrouter.ai/models
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(base_url="https://openrouter.ai/api/v1",
                            api_key=os.environ["OPENROUTER_API_KEY"]),
)
print("Ready:", MODEL_NAME)

## The Agent object

An `Agent` bundles a model, a system prompt, its tools, and (later) an output type. You create it once and reuse it. Think of it as a configured, callable assistant.

In [ ]:
agent = Agent(model, system_prompt="You are a concise travel assistant. Answer in two sentences max.")

result = await agent.run("I have one day in Lisbon. What should I not miss?")
print(result.output)

```{tip}
In a notebook, always use `await agent.run(...)`. PydanticAI's `agent.run_sync(...)` raises `This event loop is already running` inside Jupyter/Colab, because the notebook is already inside an event loop. (`run_sync` is for plain scripts.)
```

```{note}
In standalone Python scripts (outside notebooks), use `agent.run_sync(...)` instead — no `await` needed. The async `await agent.run(...)` is for notebooks and async frameworks like FastAPI (which you'll use in Block 3).
```

```{note}
The current PydanticAI docs increasingly use `instructions=` (and `@agent.instructions`) rather than `system_prompt=`. Both work; the difference matters only with `message_history` (1.4): `instructions` from earlier runs are **not** re-sent, while a `system_prompt` message rides along in the history. We use `system_prompt=` throughout for consistency — if the docs show `instructions=`, it's the same idea.
```

## The result object

`result.output` is the answer. The result also carries the full message history and usage — the same things you inspected by hand in 0.1, now on a typed object.

In [ ]:
print("output:", result.output)
print("usage: ", result.usage())         # tokens, as in 0.1
print("messages:", len(result.all_messages()), "messages in the run")

## Static vs dynamic system prompts

The string you pass is a **static** system prompt. When the instructions depend on something computed at runtime (the date, the user's name), use a `@agent.system_prompt` function — it runs on every call and its return value is added to the system prompt.

In [ ]:
from datetime import date

agent = Agent(model, system_prompt="You are a helpful assistant.")

@agent.system_prompt
def add_today() -> str:
    return f"Today's date is {date.today().isoformat()}."

result = await agent.run("What's today's date, and is it a weekday?")
print(result.output)

```{note}
This solves a real problem: a static prompt can't know today's date, but the model's training data is frozen in the past. A dynamic system prompt is your first tool for **context engineering** — putting the right facts in front of the model at the right time.
```

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Reuse the agent.** Call `await agent.run(...)` a few times with different questions. One configured agent, many runs.
2. **Make it terse.** Change the system prompt to demand one-word answers and watch behavior change with zero code changes.
3. **Add another dynamic fact.** Add a second `@agent.system_prompt` that injects a fake "user's city" and ask a location-dependent question.
::::

**What's next:** in **1.2** we give the agent **tools** and learn why the tool *description* is the single biggest lever on agent quality.